In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import random

# --- 1. Generate Synthetic Personal Data ---
personal_data = pd.DataFrame({
    'person_id': range(1, 11),
    'name': [f'Person_{i}' for i in range(1, 11)],
    'age': [random.randint(25, 60) for _ in range(10)],
    'gender': random.choices(['Male', 'Female'], k=10)
})
display(personal_data.head())

In [ ]:
# --- 2. Generate Synthetic Job Data ---
job_data = pd.DataFrame({
   'job_id': range(101, 111),
    'person_id': random.sample(range(1, 11), 10), # Link to personal_data
    'job_title': random.choices(['Software Engineer', 'Data Analyst', 'Project Manager', 'HR Specialist', 'Marketing Lead'], k=10),
    'department': random.choices(['Engineering', 'Data Science', 'Operations', 'Human Resources', 'Marketing'], k=10),
    'start_date': pd.to_datetime(['2020-01-15', '2019-03-01', '2021-06-20', '2022-02-10', '2018-09-01', '2023-01-01', '2020-11-11', '2021-04-01', '2017-07-07', '2022-08-15'])
})
display(job_data.head())

Now, let's combine these two datasets to build a knowledge graph. We'll treat `person_id` as the common link between individuals and their jobs. The graph will show people and their associated job titles and departments.

In [ ]:
# --- 3. Build the Knowledge Graph ---

G = nx.Graph()

# Add nodes for people
for index, row in personal_data.iterrows():
    G.add_node(f"Person_{row['person_id']}", type='person', name=row['name'], age=row['age'], gender=row['gender'])

# Add nodes for jobs, departments, and titles, and establish relationships
for index, row in job_data.iterrows():
    person_node = f"Person_{row['person_id']}"
    job_node = f"Job_{row['job_id']}"
    title_node = f"Title_{row['job_title']}"
    department_node = f"Dept_{row['department']}"

    G.add_node(job_node, type='job', job_id=row['job_id'], start_date=row['start_date'])
    G.add_node(title_node, type='job_title', name=row['job_title'])
    G.add_node(department_node, type='department', name=row['department'])

    G.add_edge(person_node, job_node, relation='holds_job')
    G.add_edge(job_node, title_node, relation='has_title')
    G.add_edge(job_node, department_node, relation='in_department')

print(f"Number of nodes in the graph: {G.number_of_nodes()}")
print(f"Number of edges in the graph: {G.number_of_edges()}")

Now let's visualize the graph. We'll use different colors and shapes to distinguish between people, jobs, titles, and departments for a clearer representation. Due to the complexity of 3D 'brain-like' visualizations, this will be a 2D representation.

In [ ]:
# --- 4. Visualize the Knowledge Graph (2D) ---

plt.figure(figsize=(15, 12))

# Define node colors and shapes based on type
node_colors = []
node_sizes = []
node_labels = {}

for node in G.nodes():
    node_type = G.nodes[node]['type']
    if node_type == 'person':
        node_colors.append('skyblue')
        node_sizes.append(1000)
        node_labels[node] = G.nodes[node]['name']
    elif node_type == 'job':
        node_colors.append('lightcoral')
        node_sizes.append(600)
        node_labels[node] = f"Job {G.nodes[node]['job_id']}"
    elif node_type == 'job_title':
        node_colors.append('lightgreen')
        node_sizes.append(400)
        node_labels[node] = G.nodes[node]['name']
    elif node_type == 'department':
        node_colors.append('gold')
        node_sizes.append(400)
        node_labels[node] = G.nodes[node]['name']

# Use a spring layout for better visualization of connections
pos = nx.spring_layout(G, k=0.8, iterations=50)

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9)

# Draw edges
nx.draw_networkx_edges(G, pos, edge_color='gray', alpha=0.6)

# Draw labels
nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=8, font_weight='bold')

plt.title("Synthetic Knowledge Graph (People, Jobs, Titles, Departments)", size=15)
plt.axis('off')
plt.show()

In [ ]:
# Install plotly for interactive 3D visualization if not already installed
!pip install plotly
import plotly.graph_objects as go

Now, let's create a 3D layout for our graph and visualize it using Plotly. This will allow for interactive exploration.

In [ ]:
# --- 5. Visualize the Knowledge Graph (Interactive 3D) ---

# Calculate 3D positions for nodes using a spring layout
pos_3d = nx.spring_layout(G, dim=3, k=0.8, iterations=50)

# Extract node coordinates
x_nodes = [pos_3d[node][0] for node in G.nodes()]
y_nodes = [pos_3d[node][1] for node in G.nodes()]
z_nodes = [pos_3d[node][2] for node in G.nodes()]

# Create edge traces
edge_x = []
edge_y = []
edge_z = []
for edge in G.edges():
    x0, y0, z0 = pos_3d[edge[0]]
    x1, y1, z1 = pos_3d[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])
    edge_z.extend([z0, z1, None])

edge_trace = go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    line=dict(width=0.5, color='#888'),
    hoverinfo='none',
    mode='lines')

# Create node traces with colors and labels
node_colors_map = {
    'person': 'skyblue',
    'job': 'lightcoral',
    'job_title': 'lightgreen',
    'department': 'gold'
}

node_sizes_map = {
    'person': 10,
    'job': 8,
    'job_title': 6,
    'department': 6
}

node_trace = go.Scatter3d(
    x=x_nodes, y=y_nodes, z=z_nodes,
    mode='markers+text',
    hoverinfo='text',
    marker=dict(
        symbol='circle',
        size=[node_sizes_map[G.nodes[node]['type']] for node in G.nodes()],
        color=[node_colors_map[G.nodes[node]['type']] for node in G.nodes()],
        line=dict(color='black', width=0.5)
    ),
    text=[node_labels[node] for node in G.nodes()],
    textposition='top center'
)

# Create the 3D figure
fig = go.Figure(data=[edge_trace, node_trace],
                layout=go.Layout(
                    title='Interactive 3D Knowledge Graph',
                    showlegend=False,
                    hovermode='closest',
                    scene=dict(
                        xaxis=dict(showbackground=False, showticklabels=False, zeroline=False, title=''),
                        yaxis=dict(showbackground=False, showticklabels=False, zeroline=False, title=''),
                        zaxis=dict(showbackground=False, showticklabels=False, zeroline=False, title=''),
                        aspectmode='cube'
                    )
                ))

fig.show()

Let's add a search functionality to the 3D graph. You can input a search term, and any matching nodes (and their immediate neighbors) will be highlighted.

In [ ]:
# --- 6. Add Search and Highlight Functionality to 3D Graph ---

def search_and_highlight_graph(graph, search_term, pos_3d, original_node_labels, node_colors_map, node_sizes_map):
    highlighted_nodes = set()
    highlighted_edges = set()

    # Find nodes that match the search term
    for node in graph.nodes():
        node_attributes = graph.nodes[node]
        # Search in name for person nodes, name for job_title/department nodes
        if node_attributes['type'] == 'person' and search_term.lower() in node_attributes['name'].lower():
            highlighted_nodes.add(node)
        elif node_attributes['type'] == 'job_title' and search_term.lower() in node_attributes['name'].lower():
            highlighted_nodes.add(node)
        elif node_attributes['type'] == 'department' and search_term.lower() in node_attributes['name'].lower():
            highlighted_nodes.add(node)
        # For job nodes, we can search by job_id or job_title of connected nodes
        elif node_attributes['type'] == 'job' and search_term.lower() in str(node_attributes['job_id']).lower():
            highlighted_nodes.add(node)

    # Also highlight direct neighbors of the found nodes and the connecting edges
    for h_node in list(highlighted_nodes): # Convert to list to iterate while modifying
        for neighbor in graph.neighbors(h_node):
            highlighted_nodes.add(neighbor)
            highlighted_edges.add(tuple(sorted((h_node, neighbor)))) # Add sorted tuple to avoid duplicates

    # --- Re-create Node Traces with highlighting ---
    x_nodes = [pos_3d[node][0] for node in graph.nodes()]
    y_nodes = [pos_3d[node][1] for node in graph.nodes()]
    z_nodes = [pos_3d[node][2] for node in graph.nodes()]

    current_node_colors = []
    current_node_sizes = []
    for node in graph.nodes():
        if node in highlighted_nodes:
            current_node_colors.append('red') # Highlight color
            current_node_sizes.append(node_sizes_map[graph.nodes[node]['type']] * 1.5) # Make highlighted nodes larger
        else:
            current_node_colors.append(node_colors_map[graph.nodes[node]['type']])
            current_node_sizes.append(node_sizes_map[graph.nodes[node]['type']])

    node_trace_highlighted = go.Scatter3d(
        x=x_nodes, y=y_nodes, z=z_nodes,
        mode='markers+text',
        hoverinfo='text',
        marker=dict(
            symbol='circle',
            size=current_node_sizes,
            color=current_node_colors,
            line=dict(color='black', width=0.5)
        ),
        text=[original_node_labels[node] for node in graph.nodes()],
        textposition='top center'
    )

    # --- Re-create Edge Traces with highlighting ---
    edge_x_highlighted = []
    edge_y_highlighted = []
    edge_z_highlighted = []
    edge_colors_highlighted = []

    for edge in graph.edges():
        x0, y0, z0 = pos_3d[edge[0]]
        x1, y1, z1 = pos_3d[edge[1]]
        edge_x_highlighted.extend([x0, x1, None])
        edge_y_highlighted.extend([y0, y1, None])
        edge_z_highlighted.extend([z0, z1, None])

        if tuple(sorted(edge)) in highlighted_edges:
            edge_colors_highlighted.extend(['red', 'red', 'red']) # Highlighted edge color
        else:
            edge_colors_highlighted.extend(['#888', '#888', '#888']) # Original edge color

    # Correctly create a list of colors for each point (x0, y0, z0), (x1, y1, z1), (None) triplets
    # Plotly's line color works differently, we need one color for the entire trace or segment
    # For simplicity, we'll create separate traces for highlighted and non-highlighted edges

    # Non-highlighted edges
    non_highlighted_edge_x = []
    non_highlighted_edge_y = []
    non_highlighted_edge_z = []
    highlighted_edge_x_only = []
    highlighted_edge_y_only = []
    highlighted_edge_z_only = []

    for edge in graph.edges():
        x0, y0, z0 = pos_3d[edge[0]]
        x1, y1, z1 = pos_3d[edge[1]]
        if tuple(sorted(edge)) in highlighted_edges:
            highlighted_edge_x_only.extend([x0, x1, None])
            highlighted_edge_y_only.extend([y0, y1, None])
            highlighted_edge_z_only.extend([z0, z1, None])
        else:
            non_highlighted_edge_x.extend([x0, x1, None])
            non_highlighted_edge_y.extend([y0, y1, None])
            non_highlighted_edge_z.extend([z0, z1, None])

    edge_trace_non_highlighted = go.Scatter3d(
        x=non_highlighted_edge_x, y=non_highlighted_edge_y, z=non_highlighted_edge_z,
        line=dict(width=0.5, color='#888'),
        hoverinfo='none',
        mode='lines')

    edge_trace_highlighted = go.Scatter3d(
        x=highlighted_edge_x_only, y=highlighted_edge_y_only, z=highlighted_edge_z_only,
        line=dict(width=2, color='red'), # Thicker, red for highlighted
        hoverinfo='none',
        mode='lines')

    # Create the 3D figure with highlighting
    fig = go.Figure(data=[edge_trace_non_highlighted, edge_trace_highlighted, node_trace_highlighted],
                    layout=go.Layout(
                        title=f'Interactive 3D Knowledge Graph (Searching for: "{search_term}")',
                        showlegend=False,
                        hovermode='closest',
                        scene=dict(
                            xaxis=dict(showbackground=False, showticklabels=False, zeroline=False, title=''),
                            yaxis=dict(showbackground=False, showticklabels=False, zeroline=False, title=''),
                            zaxis=dict(showbackground=False, showticklabels=False, zeroline=False, title=''),
                            aspectmode='cube'
                        )
                    ))

    fig.show()

# Example usage:
# You can change the search_query below to find different elements
search_query = 'Person_7' # Try 'Person_1', 'Manager', 'Marketing', '101', 'Engineer', 'HR'
search_and_highlight_graph(G, search_query, pos_3d, node_labels, node_colors_map, node_sizes_map)


In [ ]:
# --- Save the graph to disk ---
output_file = 'knowledge_graph.graphml'

# Convert Timestamp objects to string for GraphML compatibility
# GraphML does not support pandas.Timestamp directly.
for node, data in G.nodes(data=True):
    if data.get('type') == 'job' and 'start_date' in data:
        data['start_date'] = str(data['start_date'])

nx.write_graphml(G, output_file)
print(f"Graph saved to {output_file}")

# You can also load it back later with:
# G_loaded = nx.read_graphml(output_file)


---

In [ ]:
import networkx as nx

# Load the graph back from the GraphML file
G_loaded = nx.read_graphml('knowledge_graph.graphml')

print(f"Loaded graph has {G_loaded.number_of_nodes()} nodes and {G_loaded.number_of_edges()} edges.")

# You can then inspect the loaded graph, for example, by looking at its nodes and their attributes.
# For instance, let's see the attributes of a few nodes:
for i, node in enumerate(G_loaded.nodes(data=True)):
    print(node)
    if i > 2: # Print only first 3 nodes for brevity
        break